In [2]:
# pip install langchain_chroma

In [3]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

load_dotenv()

True

In [4]:
collection_name = "demo_1"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory="chroma_db"
)


In [5]:
# load the data
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
pages = loader.load()

/Users/rahultiwari/Documents/wills_18th_july_batch/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
pages

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-28T12:50:55+00:00', 'author': 'BajajBot Training Data', 'keywords': '', 'moddate': '2026-05-28T12:50:55+00:00', 'subject': '(unspecified)', 'title': 'Bajaj Finance Helpdesk Knowledge Base — Prose Edition', 'trapped': '/False', 'source': 'bajaj_finance_policy_prose_v1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='BAJAJ FINANCE LIMITED\n Helpdesk Agent Knowledge Base\n Prose Reference Edition — FY 2024–25\n Document Type: Internal Training & Reference Manual\n Coverage: Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy\n Intended Users: Helpdesk Agents · Branch Executives · Collections Team\n Classification: CONFIDENTIAL — For Internal Use Only\n Version: v1.0 Prose Edition — May 2025\nThis document is the prose-format reference edition of the Bajaj Finance Helpdesk Knowledge Base. All\npolicy information is presented in d

In [7]:
# lets define the chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len
)


# 3. Create chunks (returns a list of strings)
chunks = text_splitter.split_documents(pages)


In [9]:
len(chunks)

43

In [23]:
from uuid import uuid4

for doc in chunks:
    doc.metadata["id"] = str(uuid4())

In [26]:
# chunks[0].metadata

In [29]:
document_id = vector_store.add_documents(chunks)

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_56454/307347639.py:1: RuntimeWarning: coroutine 'VectorStore.aadd_documents' was never awaited
  document_id = vector_store.add_documents(chunks)


In [30]:
vector_store.get()

{'ids': ['88e92306-9ed2-4f2e-9a7d-955878d3f1d3',
  '52a6831c-1e4a-46c7-b3f6-46685769b260',
  '83daa264-5964-4c46-9ede-f01b59b13c5d',
  '91553742-b6e4-465d-a06d-2d73b2ad7d31',
  '29a0a8cd-9417-4312-b86e-98432bec52b2',
  '11e9f1df-d71f-483a-abd0-3a0c1b5f1849',
  '356c7e45-f286-4a8f-84df-0972071bcb98',
  '662b718d-f40a-41dc-922d-92a997f1c510',
  '38564612-d583-49e5-bcfb-c2c8eb2cbcb2',
  '56979dcc-5073-445c-8a9c-c32ab235d94f',
  'd9d61acb-7b3d-4d56-ab17-794f6820c0ff',
  '175e3d00-01f3-459d-8f89-6d51aa08a7c3',
  'eedee4c8-9185-42cc-9aa3-04e3a64805b2',
  '5fe285f8-0c3b-4a39-a9c5-d069899d0a1e',
  '5b45272c-bdfe-4450-9841-3721140038b0',
  'fdb6f2b0-ef1d-45ef-adf2-7c8502e368e2',
  '1cdd0ef4-930b-46b3-b71b-ce986e5e7a6b',
  '3be56151-31be-4047-a01e-cf26e9fa8098',
  'ffd7acca-f529-473a-b8ce-0e1fa8794082',
  'aab1fddf-05df-495a-8e33-4daea5fa3afc',
  '20ec563c-a7f1-4ab6-afd7-e33dccbec7e6',
  'b1369c15-055b-4563-8980-044047f06f00',
  '9d20facb-c06f-48e0-acb3-81e7b84033f3',
  '7359e5ea-1ed7-457b-93b6-

In [ ]:
retrieval = vector_store.as_retriever(search_kwargs={"k": 5})

query = "how to make a cake"

results = retrieval.invoke(query)

In [37]:
results

[Document(id='5768fc05-0838-4dd2-afb3-f14240a07005', metadata={'moddate': '2026-05-28T12:50:55+00:00', 'id': '0e442980-9e13-4a50-b515-3915181043b4', 'total_pages': 14, 'page': 7, 'page_label': '8', 'source': 'bajaj_finance_policy_prose_v1.pdf', 'title': 'Bajaj Finance Helpdesk Knowledge Base — Prose Edition', 'subject': '(unspecified)', 'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-28T12:50:55+00:00', 'author': 'BajajBot Training Data', 'trapped': '/False', 'keywords': ''}, page_content='duration of the holiday, and the customer is notified in writing. All applications must be made in person\nat a branch — online requests are not accepted for this facility.'),
 Document(id='d9d61acb-7b3d-4d56-ab17-794f6820c0ff', metadata={'source': 'bajaj_finance_policy_prose_v1.pdf', 'id': 'ca087e6e-431f-4ac4-b0b3-2123cedf937e', 'producer': 'ReportLab PDF Library - (opensource)', 'page_label': '4', 'title': 'Bajaj Finance Helpdesk Knowledge Ba

In [11]:
from uuid import uuid4


In [ ]:
# Document(id='e972bef0-3274-4426-a90d-0082e478ac2e'

In [ ]:
# # lets add id to each documets
# for i in range(len(chunks)):
#     chunks[i].metadata["id"] = str(uuid4())

TypeError: 'Document' object does not support item assignment

In [ ]:
chunks.

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-28T12:50:55+00:00', 'author': 'BajajBot Training Data', 'keywords': '', 'moddate': '2026-05-28T12:50:55+00:00', 'subject': '(unspecified)', 'title': 'Bajaj Finance Helpdesk Knowledge Base — Prose Edition', 'trapped': '/False', 'source': 'bajaj_finance_policy_prose_v1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1', 'id': 'd9956445-7c19-46fd-aa4b-b7564dcf35ce'}, page_content='BAJAJ FINANCE LIMITED\n Helpdesk Agent Knowledge Base\n Prose Reference Edition — FY 2024–25\n Document Type: Internal Training & Reference Manual\n Coverage: Personal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy\n Intended Users: Helpdesk Agents · Branch Executives · Collections Team\n Classification: CONFIDENTIAL — For Internal Use Only\n Version: v1.0 Prose Edition — May 2025\nThis document is the prose-format reference edition of the Bajaj Finance Helpdesk Knowledge B